In [ ]:
# ============================================================
# STEP 13A.1 — RESTORE FINAL 10-STATION SYSTEM
# ============================================================

from google.colab import drive

import os
import json
import joblib
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_PATH = (
    "/content/drive/MyDrive/Delhi-AQI-Prediction"
)

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)

os.makedirs(
    TABLE_PATH,
    exist_ok=True
)

# Final raw 10-station dataset
raw_10_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_10_station_raw.parquet"
    )
)

raw_10_df["Datetime"] = pd.to_datetime(
    raw_10_df["Datetime"]
)

raw_10_df["StationId"] = (
    raw_10_df["StationId"]
    .astype(str)
)

raw_10_df = (
    raw_10_df
    .sort_values(
        ["StationId", "Datetime"]
    )
    .reset_index(drop=True)
)

# Final production candidate
rf_model = joblib.load(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_random_forest.joblib"
    )
)

print("=" * 60)
print("FINAL SYSTEM RESTORED")
print("=" * 60)

print(
    "Raw records:",
    len(raw_10_df)
)

print(
    "Stations:",
    raw_10_df["StationId"].nunique()
)

print(
    "Model:",
    "Random Forest"
)

In [ ]:
# ============================================================
# STEP 13A.2 — CANONICAL FEATURE CONTRACT
# ============================================================

RAW_FEATURES = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

TARGET = "AQI_t_plus_1"

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{column}_lag_{lag}h"
    for lag in [1, 2, 12]
    for column in RAW_FEATURES
]

rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

# Remove accidental duplicate if present
rolling_features = list(
    dict.fromkeys(
        rolling_features
    )
)

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

numeric_features = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
)

model_features = (
    numeric_features
    + ["StationId"]
)

print(
    "Total model features:",
    len(model_features)
)

In [ ]:
# ============================================================
# STEP 13A.3 — CANONICAL FEATURE BUILDER
# ============================================================

DIWALI_DATES = pd.to_datetime([
    "2015-11-11",
    "2016-10-30",
    "2017-10-19",
    "2018-11-07",
    "2019-10-27",
    "2020-11-14"
])


def is_diwali_window(
    timestamp,
    days_before=2,
    days_after=2
):
    timestamp = pd.Timestamp(
        timestamp
    )

    return int(
        any(
            (
                timestamp
                >= date
                - pd.Timedelta(
                    days=days_before
                )
            )
            and
            (
                timestamp
                <= date
                + pd.Timedelta(
                    days=days_after
                )
            )
            for date in DIWALI_DATES
        )
    )


def build_canonical_features(
    history_df
):
    df = history_df.copy()

    df["Datetime"] = pd.to_datetime(
        df["Datetime"]
    )

    df["StationId"] = (
        df["StationId"]
        .astype(str)
    )

    df = (
        df[
            [
                "StationId",
                "StationName",
                "Datetime"
            ]
            + RAW_FEATURES
        ]
        .sort_values(
            ["StationId", "Datetime"]
        )
        .drop_duplicates(
            ["StationId", "Datetime"],
            keep="first"
        )
        .reset_index(drop=True)
    )

    output = []

    for station_id, station in df.groupby(
        "StationId",
        sort=False
    ):

        station = (
            station
            .sort_values("Datetime")
            .reset_index(drop=True)
            .copy()
        )

        # Calendar
        station["Hour"] = (
            station["Datetime"].dt.hour
        )

        station["Day"] = (
            station["Datetime"].dt.day
        )

        station["Month"] = (
            station["Datetime"].dt.month
        )

        station["Year"] = (
            station["Datetime"].dt.year
        )

        station["DayOfWeek"] = (
            station["Datetime"].dt.dayofweek
        )

        station["IsWeekend"] = (
            station["DayOfWeek"] >= 5
        ).astype("int8")

        station["IsStubbleSeason"] = (
            station["Month"].isin([10, 11])
        ).astype("int8")

        station["IsDiwaliWindow"] = (
            station["Datetime"]
            .apply(is_diwali_window)
            .astype("int8")
        )

        # Lags
        for lag in [1, 2, 12]:

            lag_source = station[
                ["Datetime"] + RAW_FEATURES
            ].copy()

            lag_source["Datetime"] = (
                lag_source["Datetime"]
                + pd.Timedelta(
                    hours=lag
                )
            )

            lag_source = lag_source.rename(
                columns={
                    column:
                    f"{column}_lag_{lag}h"
                    for column in RAW_FEATURES
                }
            )

            station = station.merge(
                lag_source[
                    ["Datetime"]
                    + [
                        f"{column}_lag_{lag}h"
                        for column in RAW_FEATURES
                    ]
                ],
                on="Datetime",
                how="left",
                validate="one_to_one"
            )

        # Rolling
        rolling_source = (
            station[
                [
                    "Datetime",
                    "AQI",
                    "PM2.5",
                    "PM10",
                    "NO2"
                ]
            ]
            .drop_duplicates(
                "Datetime"
            )
            .set_index("Datetime")
            .sort_index()
        )

        for column in [
            "AQI",
            "PM2.5",
            "PM10",
            "NO2"
        ]:

            station[
                f"{column}_rolling_mean_3h"
            ] = (
                rolling_source[column]
                .rolling(
                    "3h",
                    min_periods=1
                )
                .mean()
                .to_numpy()
            )

            station[
                f"{column}_rolling_std_3h"
            ] = (
                rolling_source[column]
                .rolling(
                    "3h",
                    min_periods=2
                )
                .std()
                .to_numpy()
            )

            station[
                f"{column}_rolling_mean_12h"
            ] = (
                rolling_source[column]
                .rolling(
                    "12h",
                    min_periods=1
                )
                .mean()
                .to_numpy()
            )

            station[
                f"{column}_rolling_std_12h"
            ] = (
                rolling_source[column]
                .rolling(
                    "12h",
                    min_periods=2
                )
                .std()
                .to_numpy()
            )

        # t+1 target
        future_target = station[
            [
                "Datetime",
                "AQI"
            ]
        ].copy()

        future_target["Datetime"] = (
            future_target["Datetime"]
            - pd.Timedelta(
                hours=1
            )
        )

        future_target = (
            future_target
            .rename(
                columns={
                    "AQI": TARGET
                }
            )
            [
                [
                    "Datetime",
                    TARGET
                ]
            ]
        )

        station = station.merge(
            future_target,
            on="Datetime",
            how="left",
            validate="one_to_one"
        )

        output.append(
            station
        )

    result = pd.concat(
        output,
        ignore_index=True
    )

    return (
        result
        .sort_values(
            ["StationId", "Datetime"]
        )
        .reset_index(drop=True)
    )

print(
    "Canonical feature builder restored."
)

In [ ]:
# ============================================================
# STEP 13A.4 — LIVE ADAPTER
# ============================================================

FINAL_STATIONS = [
    "DL031",
    "DL002",
    "DL021",
    "DL028",
    "DL019",
    "DL010",
    "DL034",
    "DL007",
    "DL013",
    "DL004"
]


def validate_live_input(
    df
):
    required = [
        "StationId",
        "Datetime"
    ] + RAW_FEATURES

    missing = [
        column
        for column in required
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing live fields: {missing}"
        )

    x = df.copy()

    x["StationId"] = (
        x["StationId"]
        .astype(str)
    )

    x["Datetime"] = pd.to_datetime(
        x["Datetime"],
        errors="coerce"
    )

    if x["Datetime"].isna().any():
        raise ValueError(
            "Invalid live timestamps detected."
        )

    numeric_columns = [
        column
        for column in RAW_FEATURES
    ]

    for column in numeric_columns:
        x[column] = pd.to_numeric(
            x[column],
            errors="coerce"
        )

    unexpected = set(
        x["StationId"]
    ) - set(FINAL_STATIONS)

    if unexpected:
        raise ValueError(
            "Unexpected stations: "
            f"{unexpected}"
        )

    # Negative measurements are invalid here.
    for column in numeric_columns:

        bad = (
            x[column].notna()
            &
            x[column].lt(0)
        )

        if bad.any():
            raise ValueError(
                f"Negative values in {column}"
            )

    return (
        x
        .sort_values(
            ["StationId", "Datetime"]
        )
        .drop_duplicates(
            ["StationId", "Datetime"],
            keep="last"
        )
        .reset_index(drop=True)
    )

In [ ]:
# ============================================================
# STEP 13A.5 — HISTORICAL LIVE CLOCK
# ============================================================

simulation_times = np.sort(
    raw_10_df["Datetime"]
    .dropna()
    .unique()
)

# Use the final historical portion so the
# replay behaves like a fresh deployment.
replay_start_index = int(
    len(simulation_times) * 0.90
)

replay_times = (
    simulation_times[
        replay_start_index:
    ]
)

print(
    "Replay start:",
    replay_times.min()
)

print(
    "Replay end:",
    replay_times.max()
)

print(
    "Replay timestamps:",
    len(replay_times)
)

In [ ]:
# ============================================================
# STEP 13A.6 — OFFLINE LIVE REPLAY
# ============================================================

rng = np.random.default_rng(
    42
)

# We need timestamps that have a true t+1 observation.
eligible_times = []

max_time = raw_10_df["Datetime"].max()

for timestamp in replay_times:

    if (
        timestamp
        + pd.Timedelta(hours=1)
        <= max_time
    ):
        eligible_times.append(
            timestamp
        )

eligible_times = np.array(
    eligible_times,
    dtype="datetime64[ns]"
)

sample_size = min(
    100,
    len(eligible_times)
)

selected_times = rng.choice(
    eligible_times,
    size=sample_size,
    replace=False
)

selected_times = sorted(
    pd.to_datetime(
        selected_times
    )
)

replay_results = []

for prediction_time in selected_times:

    for station_id in FINAL_STATIONS:

        # ----------------------------------------------------
        # Only data available at t
        # ----------------------------------------------------

        history = raw_10_df[
            (
                raw_10_df["StationId"]
                == station_id
            )
            &
            (
                raw_10_df["Datetime"]
                <= prediction_time
            )
        ].copy()

        if history.empty:
            continue

        # ----------------------------------------------------
        # Validate as live input
        # ----------------------------------------------------

        try:

            history = validate_live_input(
                history
            )

        except Exception:
            continue

        # ----------------------------------------------------
        # Build canonical live features
        # ----------------------------------------------------

        canonical = (
            build_canonical_features(
                history
            )
        )

        current = canonical[
            canonical["Datetime"]
            == prediction_time
        ].copy()

        if len(current) != 1:
            continue

        X_live = current[
            model_features
        ].copy()

        # ----------------------------------------------------
        # Predict t+1
        # ----------------------------------------------------

        prediction = float(
            rf_model.predict(
                X_live
            )[0]
        )

        # ----------------------------------------------------
        # Reveal actual future value only AFTER prediction
        # ----------------------------------------------------

        actual_row = raw_10_df[
            (
                raw_10_df["StationId"]
                == station_id
            )
            &
            (
                raw_10_df["Datetime"]
                ==
                prediction_time
                + pd.Timedelta(hours=1)
            )
        ]

        if len(actual_row) != 1:
            continue

        actual = float(
            actual_row[
                "AQI"
            ].iloc[0]
        )

        replay_results.append(
            {
                "StationId": station_id,
                "PredictionTime": prediction_time,
                "PredictedAQI_t_plus_1": prediction,
                "ActualAQI_t_plus_1": actual,
                "AbsoluteError": abs(
                    actual - prediction
                )
            }
        )

replay_results = pd.DataFrame(
    replay_results
)

print("=" * 60)
print("OFFLINE LIVE REPLAY COMPLETE")
print("=" * 60)

print(
    "Predictions:",
    len(replay_results)
)

print(
    "Stations:",
    replay_results[
        "StationId"
    ].nunique()
)

In [ ]:
# ============================================================
# STEP 13A.7 — REPLAY PERFORMANCE
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

replay_actual = replay_results[
    "ActualAQI_t_plus_1"
].to_numpy(
    dtype=float
)

replay_prediction = replay_results[
    "PredictedAQI_t_plus_1"
].to_numpy(
    dtype=float
)

valid = (
    np.isfinite(
        replay_actual
    )
    &
    np.isfinite(
        replay_prediction
    )
)

replay_actual = replay_actual[
    valid
]

replay_prediction = replay_prediction[
    valid
]

replay_mae = mean_absolute_error(
    replay_actual,
    replay_prediction
)

replay_rmse = np.sqrt(
    mean_squared_error(
        replay_actual,
        replay_prediction
    )
)

replay_r2 = r2_score(
    replay_actual,
    replay_prediction
)

replay_accuracy_10 = (
    np.mean(
        np.abs(
            replay_actual
            - replay_prediction
        ) <= 10
    )
    * 100
)

replay_accuracy_20 = (
    np.mean(
        np.abs(
            replay_actual
            - replay_prediction
        ) <= 20
    )
    * 100
)

print("=" * 60)
print("PHASE 13A — OFFLINE LIVE REPLAY")
print("=" * 60)

print(
    "MAE:",
    round(
        replay_mae,
        4
    )
)

print(
    "RMSE:",
    round(
        replay_rmse,
        4
    )
)

print(
    "R²:",
    round(
        replay_r2,
        6
    )
)

print(
    "Accuracy ±10 AQI:",
    round(
        replay_accuracy_10,
        2
    ),
    "%"
)

print(
    "Accuracy ±20 AQI:",
    round(
        replay_accuracy_20,
        2
    ),
    "%"
)

In [ ]:
# ============================================================
# STEP 13A.8 — BATCH VS LIVE-REPLAY
# ============================================================

batch_rf_results = {
    "MAE": 3.497741,
    "RMSE": 8.457054,
    "R2": 0.995353,
    "Accuracy_±10": 93.59
}

comparison = pd.DataFrame([
    {
        "Evaluation": "Final Batch Test",
        "MAE": batch_rf_results["MAE"],
        "RMSE": batch_rf_results["RMSE"],
        "R2": batch_rf_results["R2"],
        "Accuracy_±10_AQI_%": batch_rf_results[
            "Accuracy_±10"
        ]
    },
    {
        "Evaluation": "Offline Live Replay",
        "MAE": replay_mae,
        "RMSE": replay_rmse,
        "R2": replay_r2,
        "Accuracy_±10_AQI_%": replay_accuracy_10
    }
])

display(
    comparison.round(
        {
            "MAE": 4,
            "RMSE": 4,
            "R2": 6,
            "Accuracy_±10_AQI_%": 2
        }
    )
)

In [ ]:
# ============================================================
# STEP 13A.9 — FIXED STATION-WISE REPLAY PERFORMANCE
# ============================================================

station_replay_rows = []

for station_id, group in replay_results.groupby(
    "StationId"
):

    group = group[
        [
            "ActualAQI_t_plus_1",
            "PredictedAQI_t_plus_1"
        ]
    ].copy()

    # Convert safely to numeric
    group["ActualAQI_t_plus_1"] = pd.to_numeric(
        group["ActualAQI_t_plus_1"],
        errors="coerce"
    )

    group["PredictedAQI_t_plus_1"] = pd.to_numeric(
        group["PredictedAQI_t_plus_1"],
        errors="coerce"
    )

    # Keep only valid actual/prediction pairs
    group = group[
        np.isfinite(
            group["ActualAQI_t_plus_1"]
        )
        &
        np.isfinite(
            group["PredictedAQI_t_plus_1"]
        )
    ].copy()

    if len(group) == 0:
        continue

    actual = group[
        "ActualAQI_t_plus_1"
    ].to_numpy(
        dtype=float
    )

    predicted = group[
        "PredictedAQI_t_plus_1"
    ].to_numpy(
        dtype=float
    )

    absolute_error = np.abs(
        actual - predicted
    )

    station_replay_rows.append({
        "StationId": station_id,

        "Records": len(group),

        "MAE": mean_absolute_error(
            actual,
            predicted
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        ),

        "R2": r2_score(
            actual,
            predicted
        ),

        "Accuracy_±10_AQI_%": (
            np.mean(
                absolute_error <= 10
            ) * 100
        ),

        "Accuracy_±20_AQI_%": (
            np.mean(
                absolute_error <= 20
            ) * 100
        )
    })


station_replay = pd.DataFrame(
    station_replay_rows
)

station_replay = (
    station_replay
    .sort_values(
        "RMSE"
    )
    .reset_index(drop=True)
)

station_replay = station_replay.round({
    "MAE": 4,
    "RMSE": 4,
    "R2": 6,
    "Accuracy_±10_AQI_%": 2,
    "Accuracy_±20_AQI_%": 2
})

print("=" * 80)
print("STATION-WISE OFFLINE LIVE REPLAY PERFORMANCE")
print("=" * 80)

display(
    station_replay
)

print(
    "\nStations evaluated:",
    len(station_replay)
)

In [ ]:
# ============================================================
# STEP 13A.9A — INVALID REPLAY DIAGNOSTIC
# ============================================================

invalid_replay = replay_results[
    ~(
        np.isfinite(
            pd.to_numeric(
                replay_results[
                    "ActualAQI_t_plus_1"
                ],
                errors="coerce"
            )
        )
        &
        np.isfinite(
            pd.to_numeric(
                replay_results[
                    "PredictedAQI_t_plus_1"
                ],
                errors="coerce"
            )
        )
    )
].copy()

print("=" * 70)
print("INVALID REPLAY ROWS")
print("=" * 70)

print(
    "Total replay rows:",
    len(replay_results)
)

print(
    "Invalid rows:",
    len(invalid_replay)
)

if len(invalid_replay) > 0:
    display(
        invalid_replay.head(20)
    )
else:
    print(
        "No invalid replay rows."
    )

In [ ]:
# ============================================================
# STEP 13A.10 — FIXED AQI-REGIME PERFORMANCE
# ============================================================

# ------------------------------------------------------------
# 1. Make sure numeric columns are numeric
# ------------------------------------------------------------

replay_regime = replay_results.copy()

replay_regime["ActualAQI_t_plus_1"] = pd.to_numeric(
    replay_regime["ActualAQI_t_plus_1"],
    errors="coerce"
)

replay_regime["PredictedAQI_t_plus_1"] = pd.to_numeric(
    replay_regime["PredictedAQI_t_plus_1"],
    errors="coerce"
)

# ------------------------------------------------------------
# 2. Remove invalid prediction/actual pairs
# ------------------------------------------------------------

replay_regime = replay_regime[
    np.isfinite(
        replay_regime[
            "ActualAQI_t_plus_1"
        ]
    )
    &
    np.isfinite(
        replay_regime[
            "PredictedAQI_t_plus_1"
        ]
    )
].copy()

# ------------------------------------------------------------
# 3. Define AQI regime
# ------------------------------------------------------------

def classify_aqi(value):

    if value <= 100:
        return "Low"

    elif value <= 200:
        return "Moderate"

    elif value <= 300:
        return "High"

    elif value <= 400:
        return "Very High"

    else:
        return "Extreme"


replay_regime["AQIRegime"] = (
    replay_regime[
        "ActualAQI_t_plus_1"
    ]
    .apply(classify_aqi)
)

# ------------------------------------------------------------
# 4. Calculate metrics regime by regime
# ------------------------------------------------------------

regime_rows = []

for regime, group in replay_regime.groupby(
    "AQIRegime"
):

    actual = group[
        "ActualAQI_t_plus_1"
    ].to_numpy(
        dtype=float
    )

    predicted = group[
        "PredictedAQI_t_plus_1"
    ].to_numpy(
        dtype=float
    )

    absolute_error = np.abs(
        actual - predicted
    )

    if len(actual) == 0:
        continue

    regime_rows.append(
        {
            "AQIRegime": regime,

            "Records": len(actual),

            "MAE": mean_absolute_error(
                actual,
                predicted
            ),

            "RMSE": np.sqrt(
                mean_squared_error(
                    actual,
                    predicted
                )
            ),

            "R2": (
                r2_score(
                    actual,
                    predicted
                )
                if len(actual) >= 2
                and np.std(actual) > 0
                else np.nan
            ),

            "Accuracy_±10_AQI_%": (
                np.mean(
                    absolute_error <= 10
                ) * 100
            ),

            "Accuracy_±20_AQI_%": (
                np.mean(
                    absolute_error <= 20
                ) * 100
            )
        }
    )

regime_results = pd.DataFrame(
    regime_rows
)

# Sensible AQI order
regime_order = [
    "Low",
    "Moderate",
    "High",
    "Very High",
    "Extreme"
]

regime_results["AQIRegime"] = pd.Categorical(
    regime_results["AQIRegime"],
    categories=regime_order,
    ordered=True
)

regime_results = (
    regime_results
    .sort_values("AQIRegime")
    .reset_index(drop=True)
)

# Round
regime_results = regime_results.round({
    "MAE": 4,
    "RMSE": 4,
    "R2": 6,
    "Accuracy_±10_AQI_%": 2,
    "Accuracy_±20_AQI_%": 2
})

print("=" * 80)
print("AQI-REGIME OFFLINE LIVE REPLAY PERFORMANCE")
print("=" * 80)

display(
    regime_results
)

In [ ]:
# ============================================================
# STEP 13A.10A — REGIME DATA DIAGNOSTIC
# ============================================================

print(
    "Original replay rows:",
    len(replay_results)
)

print(
    "Valid replay rows:",
    len(replay_regime)
)

print(
    "Rows excluded:",
    len(replay_results)
    - len(replay_regime)
)

print(
    "\nAQI regime counts:"
)

print(
    replay_regime[
        "AQIRegime"
    ].value_counts()
)

In [ ]:
replay_path = os.path.join(
    TABLE_PATH,
    "phase13A_offline_live_replay.csv"
)

replay_results.to_csv(
    replay_path,
    index=False
)

station_replay.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase13A_station_replay_performance.csv"
    )
)

regime_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase13A_aqi_regime_performance.csv"
    )

)

print(
    "Replay results saved."
)

In [ ]:
adapter_code = r'''
# Production live adapter interface

def process_live_observations(
    live_df,
    station_id,
    prediction_time,
    model,
    build_canonical_features,
    model_features
):
    """
    Receives observations available up to t,
    builds canonical features, and predicts AQI(t+1).
    """

    live_df = live_df.copy()

    live_df["Datetime"] = pd.to_datetime(
        live_df["Datetime"]
    )

    live_df["StationId"] = (
        live_df["StationId"].astype(str)
    )

    history = live_df[
        (
            live_df["StationId"]
            == str(station_id)
        )
        &
        (
            live_df["Datetime"]
            <= pd.Timestamp(
                prediction_time
            )
        )
    ].copy()

    if history.empty:
        raise ValueError(
            "No live history available."
        )

    canonical = build_canonical_features(
        history
    )

    current = canonical[
        canonical["Datetime"]
        == pd.Timestamp(
            prediction_time
        )
    ].copy()

    if len(current) != 1:
        raise ValueError(
            "Current timestamp is not available "
            "exactly once."
        )

    X_live = current[
        model_features
    ].copy()

    prediction = float(
        model.predict(
            X_live
        )[0]
    )

    return prediction
'''

adapter_path = os.path.join(
    OUTPUT_PATH,
    "phase13A_live_adapter.py"
)

with open(
    adapter_path,
    "w"
) as f:
    f.write(
        adapter_code
    )

print(
    "Live adapter saved:",
    adapter_path
)

In [ ]:
phase13A_summary = {
    "phase": (
        "Phase 13A - Live Adapter + Offline Replay"
    ),

    "station_count": 10,

    "model": "Random Forest",

    "forecast_target": "AQI_t_plus_1",

    "replay_predictions": int(
        len(replay_results)
    ),

    "MAE": float(
        replay_mae
    ),

    "RMSE": float(
        replay_rmse
    ),

    "R2": float(
        replay_r2
    ),

    "accuracy_plus_minus_10_AQI": float(
        replay_accuracy_10
    ),

    "accuracy_plus_minus_20_AQI": float(
        replay_accuracy_20
    ),

    "real_ogd_connected": False,

    "next_phase": (
        "Phase 13B - Random Forest Optimization"
    )
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase13A_summary.json"
)

with open(
    summary_path,
    "w"
) as f:
    json.dump(
        phase13A_summary,
        f,
        indent=4
    )

print(
    "Phase 13A summary saved."
)

In [ ]:
# ============================================================
# STEP 13A.14 — FINAL PHASE 13A REPORT
# ============================================================

print("=" * 70)
print("PHASE 13A — LIVE ADAPTER + OFFLINE REPLAY COMPLETE")
print("=" * 70)

print(
    "Stations:",
    replay_results[
        "StationId"
    ].nunique()
)

print(
    "Replay predictions:",
    len(replay_results)
)

print(
    "\nReplay MAE:",
    round(
        replay_mae,
        4
    )
)

print(
    "Replay RMSE:",
    round(
        replay_rmse,
        4
    )
)

print(
    "Replay R²:",
    round(
        replay_r2,
        6
    )
)

print(
    "Replay Accuracy ±10 AQI:",
    round(
        replay_accuracy_10,
        2
    ),
    "%"
)

print(
    "Replay Accuracy ±20 AQI:",
    round(
        replay_accuracy_20,
        2
    ),
    "%"
)

print(
    "\nReal CPCB/OGD connected:",
    False
)

print(
    "Next:",
    "Phase 13B - Random Forest Optimization"
)

print("=" * 70)

In [ ]:
# ============================================================
# PHASE 13A — CHECKPOINT
# ============================================================

import os
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)

os.makedirs(
    TABLE_PATH,
    exist_ok=True
)

phase13A_summary = {
    "completed_phase": (
        "Phase 13A - Live Adapter + Offline Replay"
    ),

    "next_phase": (
        "Phase 13B - Random Forest Optimization"
    ),

    "stations": 10,

    "replay_predictions": 1000,

    "replay_mae": 3.5976,

    "replay_rmse": 8.6435,

    "replay_r2": 0.994667,

    "accuracy_plus_minus_10_aqi": 92.80,

    "accuracy_plus_minus_20_aqi": 97.46,

    "real_cpcb_ogd_connected": False,

    "status": "Offline replay validated"
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase13A_summary.json"
)

with open(
    summary_path,
    "w"
) as f:
    json.dump(
        phase13A_summary,
        f,
        indent=4
    )

# Save replay results again for safety
if "replay_results" in globals():

    replay_results.to_csv(
        os.path.join(
            TABLE_PATH,
            "phase13A_offline_live_replay.csv"
        ),
        index=False
    )

# Save station analysis
if "station_replay" in globals():

    station_replay.to_csv(
        os.path.join(
            TABLE_PATH,
            "phase13A_station_replay_performance.csv"
        ),
        index=False
    )

# Save AQI regime analysis
if "regime_results" in globals():

    regime_results.to_csv(
        os.path.join(
            TABLE_PATH,
            "phase13A_aqi_regime_performance.csv"
        ),
        index=False
    )

# Update session status
session_status = {
    "completed_phase": "Phase 13A",
    "current_best_model": "Random Forest",
    "station_count": 10,
    "forecast": "AQI(t) -> AQI(t+1)",
    "next_phase": "Phase 13B - Random Forest Optimization",
    "real_ogd_connected": False
}

with open(
    os.path.join(
        OUTPUT_PATH,
        "session_status.json"
    ),
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

print("=" * 60)
print("PHASE 13A CHECKPOINT SAVED")
print("=" * 60)
print("Next: Phase 13B - Random Forest Optimization")